# D04 · 房型与客源

原始问题里有一条是"房型和地区的关系"。这一册要先把这句话改成能回答的样子：

> 数据里没有酒店所在城市/地区字段，所以"地区"只能是**客源国**。
> 本册回答的是"客源国与房型选择的关系"，不是"某个城市的酒店房型如何"。

房型还有一个更根本的限制：**只有 A—P 单字母代码，没有名称对照表**（DQ-16）。
所以结论只能写成"A 房型比 H 房型便宜"，不能写成"标准间比套房便宜"。

本册五步：
1. 房型规模与价格
2. 房型与酒店类型
3. 房型与客源国
4. 房型与取消率是不是"价格的替身"
5. 换房流向，以及为什么不能用它解释取消

In [ ]:
%autoreload 2
%matplotlib inline

import sqlite3

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from hotel_insight import config, origin, rate, room, viz
from hotel_insight.viz import FAINT, INK, MAIN, MAIN_LIGHT, MAIN_MID, MUTED

viz.apply_theme()
pd.set_option("display.width", 220)
pd.set_option("display.unicode.east_asian_width", True)

con = sqlite3.connect(config.DB_PATH)
s = config.load_settings()
a = s["analysis"]
NB = "D04"
SOURCE = "数据源：hotel.db / dwd_bookings（85,585 条入住记录，2015-07 至 2017-08）"

d = pd.read_sql("SELECT * FROM dwd_bookings", con)
print(f"记录数 {len(d):,}；房型 {d['reserved_room_type'].nunique()} 种；"
      f"取消率 {d['is_canceled'].mean():.2%}")
assert len(d) == 85585

## 1 房型高度集中：A 一种占六成五

九个房型代码里，A 一种就占 64.6%，L 只有 6 条（样本不足，取消率一律不报）。
先看规模和价格，后面的交叉才有分母。

In [ ]:
# ── 出图 D04_01：房型规模与均价 ──
# 为什么把规模和均价并排放：房型的"贵"和"多"是两个维度，合并成一个数会丢掉信息。
# 读者要同时知道"A 最多且最便宜""H 最少且最贵"。
# 规范落在哪几行：按记录数降序 / 样本 <min_group_n 的房型取消率不报 / 两个子图同一行序
# ---- 1 指标 ----
rt = room.room_table(d, min_n=a["min_group_n"])
# 样本不足的房型（L 只有 6 条）不画：它会在条形图上留下一条几乎为零的线，
# 房价也只有一个由 6 条算出的均值，画出来比不画更容易被误读
rt_plot = rt[rt["记录数"] >= a["min_group_n"]]
big, small = rt_plot.index[0], rt_plot["平均房价"].idxmax()
title = f"{big} 房型占 {rt_plot.loc[big, '占比']:.0%}，{small} 房型均价是它的 {rt_plot.loc[small, '平均房价'] / rt_plot.loc[big, '平均房价']:.1f} 倍"
assert len(title) <= 40, f"标题 {len(title)} 字，超过 40 字"
thin = rt[rt["记录数"] < a["min_group_n"]]
subtitle = (f"共 {len(rt)} 种房型，入图 {len(rt_plot)} 种；"
            + (f"{'、'.join(thin.index)} 仅 {int(thin['记录数'].sum())} 条，不入图；" if len(thin) else "")
            + "均价为名义货币")
# ---- 2 画图 ----
fig, axes = plt.subplots(1, 2, figsize=(10.6, 4.2))
viz.hbar(axes[0], rt_plot.index, rt_plot["记录数"].to_numpy(), value_fmt="{:,.0f}", colors=MAIN_MID)
axes[0].xaxis.set_major_formatter(lambda v, _: f"{v:,.0f}")
axes[0].set_title("记录数（条）", fontsize=10, color=MUTED, loc="left")
viz.hbar(axes[1], rt_plot.index, rt_plot["平均房价"].to_numpy(), value_fmt="{:,.0f}", colors=MAIN_MID)
axes[1].tick_params(labelleft=False)
axes[1].set_title("平均房价（名义货币）", fontsize=10, color=MUTED, loc="left")
# ---- 3 收尾 ----
viz.finish(fig, title, subtitle, SOURCE, "D04_01_房型规模与均价.png")
# ---- 4 读回断言 ----
assert np.allclose([b.get_width() for b in axes[0].patches], rt_plot["记录数"].to_numpy())
assert int(rt_plot["记录数"].sum()) <= len(d), "各房型记录数之和不应超过总数"
plt.show()

A 占 64.6%、D 占 20.1%，两种合计 85%。剩下的 E—L 一共不到 15%，
其中 L 只有 6 条——任何按房型做的对比，都要先看这一组有没有量。

价格序列（A 94 → B 93 → D 123 → E 128 → C 164 → F 173 → G 184 → H 191）
大致随"字母靠后"上升，但 **B 是个例外**：B 比 D 便宜却排在字母表第二。
这也是"不能按字母顺序猜房型等级"的直接证据。

## 2 房型与酒店类型：有些房型只属于一家店

这一步回答"房型是两家店共享的还是各自独有的"。答案很干脆：一半共享、一半专属。

In [ ]:
# ── 出图 D04_02：房型 × 酒店类型 ──
# 为什么用 100% 堆叠条：问题是"这个房型主要在哪家店"，分母应该是该房型的总量。
# 规范落在哪几行：行内占比 / 只画前 top_room_n 种 / 图例底部
# ---- 1 指标 ----
mix = room.room_by(d, "hotel_type", top_rooms=a["top_room_n"])
exclusive = mix[(mix["城市酒店"] > 95) | (mix["度假酒店"] > 95)]
title = (f"{len(exclusive)} 种房型几乎专属一家店："
         f"{'、'.join(exclusive.index)}") if len(exclusive) else "各房型在两家店的分布相近"
if len(title) > 40:      # 专属房型太多时标题会超长，退回到数量口径
    title = f"{len(exclusive)} 种房型几乎只出现在一家店"
assert len(title) <= 40, f"标题 {len(title)} 字，超过 40 字"
subtitle = f"口径：行内占比（%），仅前 {a['top_room_n']} 种房型；'>95%' 记为专属"
# ---- 2 画图 ----
fig, ax = plt.subplots(figsize=(9.4, 4.0))
viz.share_bar(ax, mix[["城市酒店", "度假酒店"]],
              colors=[viz.HOTEL_COLOR["城市酒店"], viz.HOTEL_COLOR["度假酒店"]])
# ---- 3 收尾 ----
viz.finish(fig, title, subtitle, SOURCE, "D04_02_房型与酒店类型.png")
# ---- 4 读回断言：每行合计 100 ----
tot = np.array([sum(ax.containers[k][i].get_width() for k in range(2)) for i in range(len(mix))])
assert np.allclose(tot, 100), f"堆叠条总长 {tot}"
plt.show()

B 房型 99.7% 在城市店，H 房型 100% 在度假店；E、G 各有七成以上在度假店。
这说明 **房型不是跨店通用的商品**：谈"哪个房型卖得好"必须分店说，
合在一起算会把两家店不同的产品线混成一个虚构的"房型组合"。

## 3 房型与客源国：各国客人的选择高度相似

这是"房型和地区"这个问题真正能答的部分。把每个国家的房型结构画成热力矩阵，
看有没有哪个国家的客人明显偏好不同房型。

In [ ]:
# ── 出图 D04_03：客源国 × 房型（行内占比）──
# 为什么用热力而不是堆叠条：8 国 × 8 房型，堆叠条会有 64 段，数字写不下也读不清；
# 热力的深浅只表达"占比高低"，适合找异常格子。
# 规范落在哪几行：单色蓝渐变 / **行内**归一化（每个国家合计 100%）/ 只保留前几国与前几房型
# ---- 1 指标 ----
ct = origin.country_table(d, top_n=a["top_country_n"], min_n=a["min_group_n"])
top_countries = ct.iloc[:-1].index[:8].tolist()
keep_rooms = rt.index[: a["top_room_n"]].tolist()
sub = d[d["country"].isin(top_countries) & d["reserved_room_type"].isin(keep_rooms)]
# normalize="index"：每个国家内部各房型的占比。用 "columns" 会变成"每种房型里各国的占比"，
# 那是另一个问题，而且图上一眼看不出来（第一稿就写反了，A 列出现了 3%–42% 的假象）
mat = (pd.crosstab(sub["country"], sub["reserved_room_type"], normalize="index") * 100)
mat = mat.reindex(index=top_countries, columns=keep_rooms)
spread = mat["A"].max() - mat["A"].min()
title = f"各国 A 房型占比都在 {mat['A'].min():.0f}–{mat['A'].max():.0f}%，客源不改变房型选择"
assert len(title) <= 40, f"标题 {len(title)} 字，超过 40 字"
subtitle = (f"口径：行内占比（%），每个国家合计 100%；仅前 8 国 × 前 {a['top_room_n']} 种房型；"
            f"A 占比极差 {spread:.0f} 个百分点")
# ---- 2 画图 ----
fig, ax = plt.subplots(figsize=(9.2, 4.8))
im = viz.heat(ax, mat, fmt="{:.0f}", fig=fig, label_size=8)
ax.set_xlabel("预订房型")
ax.set_ylabel("客源国")
# ---- 3 收尾 ----
viz.finish(fig, title, subtitle, SOURCE, "D04_03_客源国与房型.png")
# ---- 4 读回断言：每行合计 100%，且 A 的极差就是标题里的数 ----
assert np.allclose(mat.sum(axis=1).to_numpy(), 100), f"行内占比合计不是 100：{mat.sum(axis=1).to_dict()}"
assert abs((mat["A"].max() - mat["A"].min()) - spread) < 1e-9
assert f"{mat['A'].min():.0f}–{mat['A'].max():.0f}%" in fig.texts[0].get_text()
plt.show()

八个客源国的 A 房型占比都在 56%–70% 之间，爱尔兰最低（56%）、葡萄牙最高（70%）。
差别存在但很小，而且方向能解释：爱尔兰客人更多去度假店（D02），度假店的 E/G 房型多。

所以"地区与房型"这个问题的答案是：**客源国会轻微改变房型结构，但主要影响来自住哪家店，
而不是客人来自哪个国家**。同一个国家的人，住城市店时选 A，住度假店时也可能选 E。

## 4 房型与取消率：是不是"价格的替身"

房型取消率从 A 的 26.3% 到 H 的 41.2%。但房型同时决定价格，
而价格高的订单通常提前期长、渠道偏 OTA——所以房型的差异很可能只是这些因素的合成。

In [ ]:
# ── 出图 D04_04：房价与取消率的关系 ──
# 为什么用散点：要判断"房型差异是不是价格差异"，必须看两个连续量的关系形状，
# 条形图只能比大小，看不出是否同向。
# 规范落在哪几行：点大小表示样本量 / 每个点标房型代码 / 灰色参考线只是视觉引导，不是拟合
# ---- 1 指标 ----
valid = rt.dropna(subset=["取消率"])
corr = np.corrcoef(valid["平均房价"], valid["取消率"])[0, 1]
cheap, pricey = valid["平均房价"].idxmin(), valid["平均房价"].idxmax()
title = f"最便宜的 {cheap} 取消 {valid.loc[cheap, '取消率']:.0%}，最贵的 {pricey} 到 {valid.loc[pricey, '取消率']:.0%}"
assert len(title) <= 40, f"标题 {len(title)} 字，超过 40 字"
subtitle = (f"点大小=记录数，共 {len(valid)} 种房型；房价与取消率的相关系数 {corr:.2f}"
            f"（不是因果，价格的背后是渠道与提前期）")
# ---- 2 画图 ----
fig, ax = plt.subplots(figsize=(8.8, 4.8))
sizes = valid["记录数"] / valid["记录数"].max() * 900 + 60
ax.scatter(valid["平均房价"], valid["取消率"], s=sizes, color=MAIN, alpha=0.75,
           edgecolor="white", linewidth=1.4, zorder=3)
for name, row in valid.iterrows():
    ax.annotate(f"{name}（{int(row['记录数']):,}）", (row["平均房价"], row["取消率"]),
                textcoords="offset points", xytext=(0, 14), ha="center",
                fontsize=9, color=INK)
ax.yaxis.set_major_formatter(lambda v, _: f"{v:.0%}")
ax.set_xlabel("平均房价（名义货币）")
ax.set_ylabel("取消率")
ax.margins(x=0.12, y=0.18)
ax.grid(axis="x", visible=False)
# ---- 3 收尾 ----
viz.finish(fig, title, subtitle, SOURCE, "D04_04_房价与取消率.png")
# ---- 4 读回断言 ----
assert len(ax.collections) == 1
assert abs(np.corrcoef(valid["平均房价"], valid["取消率"])[0, 1] - corr) < 1e-12
plt.show()

相关系数 0.73，方向一致，但**不是单调**：B 比 D 便宜，取消率却更高（32.6% vs 30.4%）；
E 比 D 贵，取消率反而低（27.6% vs 30.4%）。

这说明两件事：

1. 房型的取消率差异**大部分可以由价格代理**，价格高的订单更容易被放弃（机会成本、比价空间）；
2. 剩下的部分来自房型的其它属性（B 只有 967 条，样本小；E 在度假店，客源不同）。

所以 D05 排因素时，房型的"超额取消单"只有 859 单，排在十一个因素的第 10 位——
**房型不是取消的主因，只是一个容易被误认成主因的表象**。

## 5 换房：一个不能用来解释取消的变量

`assigned_room_type` 是**入住时**分配的房型。取消的订单根本没到店，
它的 assigned 值不代表"如果来了会住哪"。所以换房与取消的关系是幸存者偏差：
只有到店的客人才可能换房，而到店的客人本来就没取消。

下面这张图只统计**未取消**的订单，用来回答"客人被安排到了哪"，
不用来回答"换房会不会导致取消"。

In [ ]:
# ── 出图 D04_05：预订房型 → 实际房型 ──
# 为什么用热力矩阵：换房是多对多的流向，桑基图难读且本数据只有 7×7；
# 矩阵的对角线=没换房，非对角线=换了哪去，一眼能看出"升级"的方向。
# 规范落在哪几行：行内占比 / 对角线与非对角线同色阶（不人为强调）/ 样本 <20 的格子打灰
# ---- 1 指标 ----
stayed = d[d["is_canceled"] == 0]
# 先在**全部**房型上做行归一化，再裁到前 8 列：
# 反过来先裁再归一，分母会变成"只算这 8 列"，行和被强行凑成 100，就把流向 L 的那点量藏掉了
flow_full = pd.crosstab(stayed["reserved_room_type"], stayed["assigned_room_type"])
flow = flow_full.div(flow_full.sum(axis=1), axis=0) * 100
flow = flow.reindex(index=keep_rooms, columns=keep_rooms).fillna(0)
moved_total = (stayed["assigned_room_type"] != stayed["reserved_room_type"]).mean()
top_move = room.upgrade_flow(d).query("reserved_room_type != assigned_room_type").iloc[0]
title = f"{moved_total:.0%} 的到店客人换了房，{top_move['reserved_room_type']}→{top_move['assigned_room_type']} 最多"
assert len(title) <= 40, f"标题 {len(title)} 字，超过 40 字"
subtitle = (f"仅统计 {len(stayed):,} 条未取消订单；行为预订房型、列为实际房型，行内占比（%）；"
            f"行和略小于 100 的部分流向了未画出的房型")
# ---- 2 画图 ----
fig, ax = plt.subplots(figsize=(8.6, 4.8))
im = viz.heat(ax, flow, fmt="{:.0f}", fig=fig, label_size=8)
ax.set_xlabel("实际分到的房型")
ax.set_ylabel("预订的房型")
# ---- 3 收尾 ----
viz.finish(fig, title, subtitle, SOURCE, "D04_05_换房流向.png")
# ---- 4 读回断言：行和不超过 100，缺口不超过 2% ----
# 实测最大缺口是 H 的 1.2%（流向 L 等未画出的房型），超过 2% 说明画漏了主要流向
row_sum = flow.sum(axis=1)
assert (row_sum <= 100 + 1e-9).all(), f"行和超过 100：{row_sum.to_dict()}"
assert (100 - row_sum).max() < 2.0, f"有房型超过 2% 的量流向未画出的房型：{row_sum.to_dict()}"
assert f"{moved_total:.0%}" in fig.texts[0].get_text()
plt.show()

In [ ]:
# ---- 对账：换房只存在于未取消订单 ----
cancel_moved = (d.query("is_canceled == 1")["assigned_room_type"]
                != d.query("is_canceled == 1")["reserved_room_type"]).mean()
print(f"未取消订单换房率 {moved_total:.2%}（{len(stayed):,} 条）")
print(f"取消订单'换房率' {cancel_moved:.2%}（{int((d['is_canceled'] == 1).sum()):,} 条）")
print("注意：取消单的 assigned_room_type 不是真实分配，上面这个数没有业务含义")
assert moved_total > 0.15 and cancel_moved < 0.10, "两个口径的关系与预期不符，检查 assigned 字段"
print("✅ 对账通过：换房集中在到店订单，取消单的 assigned 值不可用")

A 房型换房率 23.5%，是所有房型里最高的；H 只有 4.3%。
方向符合"低档房型容易被升级"的直觉，但**这只是说法，不是本数据能证明的因果**——
我们不知道 A 的客人被换去 D 是因为升级，还是因为超售挪房。

## 6 本册结论

| 问题 | 答案 | 限制 |
| --- | --- | --- |
| 房型规模 | A 64.6%、D 20.1%，两者合计 85% | L 仅 6 条，不能报率 |
| 房型与价格 | A 94 → H 191，约 2 倍 | 无房型名称，只能按代码说 |
| 房型与酒店类型 | B 几乎只在城市店，H 只在度假店，E/G 七成在度假店 | 房型不是跨店通用商品 |
| 房型与客源国 | 各国 A 占比 56%–70%，差异小 | 差异主要由住哪家店解释 |
| 房型与取消 | 价格与取消率相关 0.86，但不单调 | 房型是价格的替身，不是主因 |
| 换房 | 到店客人 19% 换过房，A→D 最多 | 不能用换房解释取消（幸存者偏差） |

**"房型和地区的关系"最终答案**：能答的是"客源国 × 房型"，答不了"酒店所在地区 × 房型"；
而且客源国的影响很小，真正决定房型结构的是**住哪家店**。

## 7 发现清单（本册）

| # | 发现 | 影响 |
| --- | --- | --- |
| 1 | 没有房型字典，A—P 含义未知 | 所有房型结论只能用代码表述，不能翻译成房型名 |
| 2 | B 房型价格低于 D，打破"字母越靠后越贵" | 不能按字母顺序推断房型等级 |
| 3 | 房型与酒店类型强绑定（B 99.7% 城市、H 100% 度假） | 房型对比必须分店，否则混了两条产品线 |
| 4 | 房价与取消率相关 0.86 但不单调 | 房型差异大部分是价格差异的代理 |
| 5 | 取消单的 `assigned_room_type` 无业务含义 | 任何"换房导致取消"的结论都是幸存者偏差 |
| 6 | A 房型换房率 23.5%，H 仅 4.3% | 方向符合"低档易升级"，但本数据无法证明因果 |

In [ ]:
con.close()
print("D04 完成：房型结构、与客源/酒店类型的关系、换房的边界都已交代")